In [50]:
import os
%pwd

'/mnt/D6A6D17EA6D15F93/AI-ML/REAL WORLD PROJECTS/NLP/End-to-End-NLP-Text-Summarization-with-Hugging-Face-Transformers'

In [51]:
import os

if os.getcwd().endswith('research'):
    os.chdir('../')
elif not os.path.exists('config/config.yaml'):
    os.chdir('/mnt/D6A6D17EA6D15F93/AI-ML/REAL WORLD PROJECTS/NLP/End-to-End-NLP-Text-Summarization-with-Hugging-Face-Transformers')

%pwd



'/mnt/D6A6D17EA6D15F93/AI-ML/REAL WORLD PROJECTS/NLP/End-to-End-NLP-Text-Summarization-with-Hugging-Face-Transformers'

In [52]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    data_path: Path
    model_path: Path
    tokenizer_path: Path
    metric_file_name: Path

In [53]:
from src.textSummarizer.constants import *
from src.textSummarizer.utils.common import read_yaml,create_directories

In [54]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_path = config.model_path,
            tokenizer_path = config.tokenizer_path,
            metric_file_name = config.metric_file_name
            
        )

        return model_evaluation_config

In [55]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_from_disk

import torch
import pandas as pd
from tqdm import tqdm

In [56]:
from transformers import AutoModelForSeq2SeqLM,AutoTokenizer
from datasets import load_from_disk
import torch
import pandas as pd
from tqdm import tqdm
import evaluate
from src.textSummarizer.entity import ModelEvaluationConfig

class ModelEvaluation:
    def __init__(self,config:ModelEvaluationConfig):
        self.config=config

    def generate_batch_sized_chunks(self,list_of_elements,batch_size):
        for i in range(0,len(list_of_elements),batch_size):
            yield list_of_elements[i: i + batch_size]

    def calculate_metric_on_test_ds(self,dataset,metric,model,tokenizer,
                                    batch_size=16,device='cuda' if torch.cuda.is_available() else"cpu",
                                    column_text='article',
                                    column_summary='highlights'):
        article_batches=list(self.generate_batch_sized_chunks(dataset[column_text],batch_size))
        target_batches=list(self.generate_batch_sized_chunks(dataset[column_summary],batch_size))

        for article_batch,target_batch in tqdm(
            zip(article_batches,target_batches),total=len(article_batches)):

            inputs=tokenizer(article_batch,max_length=1024,truncation=True,
                             padding='max_length',return_tensors='pt')
            summaries=model.generate(input_ids=inputs['input_ids'].to(device),
                                     attention_mask=inputs['attention_mask'].to(device),
                                     length_penalty=0.8,num_beams=2,max_length=128,early_stopping=True)

            decoded_summaries=[tokenizer.decode(s,skip_special_tokens=True,
                               clean_up_tokenization_spaces=True)
                for s in summaries]

            decoded_summaries=[d.replace("<n>"," ") for d in decoded_summaries]

            metric.add_batch(predictions=decoded_summaries,references=target_batch)

        score=metric.compute()
        return score

    def evaluate(self):
        device="cuda" if torch.cuda.is_available() else "cpu"
        tokenizer=AutoTokenizer.from_pretrained(self.config.tokenizer_path)
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(
            self.config.model_path,
            torch_dtype=torch.float16 if device == "cuda" else torch.float32
        ).to(device)

        dataset_samsum_pt=load_from_disk(self.config.data_path)

        rouge_names=["rouge1", "rouge2", "rougeL", "rougeLsum"]
        rouge_metric=evaluate.load('rouge')

        score=self.calculate_metric_on_test_ds(
            dataset_samsum_pt['test'][0:10],rouge_metric,model_pegasus,tokenizer,batch_size=2,column_text='dialogue',column_summary='summary'
        )
        rouge_dict={rn : score[rn] for rn in rouge_names}

        df=pd.DataFrame(rouge_dict,index=['pegasus'])
        df.to_csv(self.config.metric_file_name,index=False)

In [57]:
config = ConfigurationManager()
model_evaluation_config = config.get_model_evaluation_config()
model_evaluation_config = ModelEvaluation(config=model_evaluation_config)
model_evaluation_config.evaluate()

[2026-08-30 00:24:49,726:INFO:common:Yaml file config/config.yaml loaded successfully]
[2026-08-30 00:24:49,728:INFO:common:Yaml file params.yaml loaded successfully]
[2026-08-30 00:24:49,729:INFO:common:Created directory at : artifacts]
[2026-08-30 00:24:49,730:INFO:common:Created directory at : artifacts/model_evaluation]


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

100%|██████████| 5/5 [00:04<00:00,  1.14it/s]

[2026-08-30 00:25:04,995:INFO:rouge_scorer:Using default tokenizer.]
